# Solución secuencial CPU — ASL

David Domínguez · 23712 / Jonathan Diaz · 23873 / Luis Padilla · 23663

**Ejecución nueva directa en este notebook:** `EXP-006` / `RUN-20261118T040831Z-cpu-notebook-01`.
Cada salida corresponde al código de la celda ejecutada. No se simula `fit` ni se
sustituye por leer un historial. La corrida es adicional a las cinco mediciones
del informe: sus promedios permanecen intactos. Al final se incluyen los registros
históricos completos identificados como tales.

Kernel: **PCUDA**. Preparación de archivos/imágenes en CPU.
Entrenamiento, validación e inferencia en CPU; librerías con paralelismo interno.
Datos congelados:17,400 imágenes,RGB64×64,float32/255;train12180,val2610,test2610.
Se ejecutan15épocas completas por red,batch128,último20,sin EarlyStopping.

Las celdas operativas se extrajeron del AST del controlador medido y se separaron
por fase. Esta ejecución verifica su recorrido en notebook. Para repetir se debe
preparar otro EXP/RUN; este destino no se sobrescribe después de cerrarlo.


In [1]:
from pathlib import Path
import ast, csv, hashlib, json, os, random, subprocess, sys, time
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "docs/Proyecto 3 CUDA.pdf").is_file())
sys.path[:0] = [str(ROOT / "src"), str(ROOT / "scripts")]
EXPERIMENT = ROOT / "resultados/secuencial/EXP-006"
CONFIG = ROOT / "configuraciones/EXP-006.json"
NOTEBOOK = ROOT / "LAB3-DATASCIENCE/Lab3_ASL_Deep_Learning.ipynb"
directory = EXPERIMENT / "RUN-20261118T040831Z-cpu-notebook-01"
repetition = 1
experiment = "EXP-006"
SEED, IMG_SIZE, NUM_CLASSES = 42, 64, 29
__file__ = str(ROOT / "scripts/benchmark_cpu.py")
assert not (directory / "salida.json").exists(), "Corrida cerrada: preparar otro EXP/RUN"
from IPython.display import display, Image, Markdown
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
print("Ejecución real:", experiment, directory.name)
print("Intérprete:", sys.executable)


Ejecución real: EXP-006 RUN-20261118T040831Z-cpu-notebook-01
Intérprete: C:\Users\David\Documents\2026\S2\PARALELA\CUDA\PROYECTO-CUDA\proyectoCUDA-paralela\PCUDA\Scripts\python.exe


## 1. Configuración, registro y funciones de medición

Las funciones siguientes registran tiempos y estado real; no entrenan al definirse.

In [2]:
FIELDS = 'experimento_id,run_id,dispositivo,repeticion,fase,tipo_medicion,duracion_s,inicio_utc,fin_utc,epocas,pasos,muestras,batch_size,incluido_resumen,estado,configuracion_sha256,datos_sha256,evidencia,motivo'.split(',')

def utc():
    return datetime.now(timezone.utc).isoformat()

def sha(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def write_json(path, value):
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

def write_csv(path, rows, fields=None):
    with Path(path).open('w', encoding='utf-8', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=fields or list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)

def original_namespace():
    """Compile only named definitions, preserving their original source bodies."""
    import numpy as np
    import pandas as pd
    from PIL import Image
    namespace = dict(np=np, pd=pd, Image=Image, random=random, SEED=42, IMG_SIZE=64)
    notebook = json.loads(NOTEBOOK.read_text(encoding='utf-8'))
    names = {'build_sampled_file_list', 'load_images', 'build_cnn_a', 'build_cnn_b', 'build_fc'}
    for index, cell in enumerate(notebook['cells']):
        if cell['cell_type'] != 'code':
            continue
        tree = ast.parse(''.join(cell['source']))
        for node in tree.body:
            if isinstance(node, ast.FunctionDef) and node.name in names:
                exec(compile(ast.Module(body=[node], type_ignores=[]), f'original/celda-{index}', 'exec'), namespace)
    namespace['_notebook'] = notebook
    return namespace

class Recorder:

    def __init__(self, directory, config, repetition):
        self.directory, self.config, self.repetition = (directory, config, repetition)
        self.rows = []
        self.epochs = []

    def state(self, phase, **more):
        write_json(self.directory / 'estado.json', dict(utc=utc(), fase=phase, **more))
        print(f'[{utc()}] {phase} {more}', flush=True)

    def measure(self, phase, function, **load):
        self.state(phase)
        start_utc, start = (utc(), time.perf_counter_ns())
        value = function()
        end, end_utc = (time.perf_counter_ns(), utc())
        self.add(phase, (end - start) / 1000000000.0, start_utc, end_utc, **load)
        return value

    def add(self, phase, seconds, start, end, **load):
        row = dict.fromkeys(FIELDS, '')
        row.update(experimento_id=self.config['experimento_id'], run_id=self.directory.name, dispositivo='cpu', repeticion=self.repetition, fase=phase, tipo_medicion='pared', duracion_s=seconds, inicio_utc=start, fin_utc=end, incluido_resumen='true', estado='completada', configuracion_sha256=sha(CONFIG), datos_sha256=self.config['datos']['manifest_splits_sha256'], evidencia=(self.directory / 'tiempos.csv').relative_to(ROOT).as_posix())
        row.update(load)
        self.rows.append(row)
        write_csv(self.directory / 'tiempos.csv', self.rows, FIELDS)
        print(f'RESULTADO {phase}: {seconds:.6f} s', flush=True)

In [3]:
config = json.loads(CONFIG.read_text(encoding="utf8"))
print(json.dumps(config, indent=2, ensure_ascii=False))
from registro_notebook import iniciar_monitor
monitor_stop, monitor_thread = iniciar_monitor(directory)


{
  "es_plantilla": false,
  "experimento_id": "EXP-006",
  "creado_utc": "2026-11-18T04:08:31.049328+00:00",
  "pregunta": "Tiempo CPU de las operaciones de procesamiento del laboratorio original completo, con carga fija.",
  "responsable": "Codex por solicitud del usuario; revision docente de codigo y salidas nativas",
  "requisitos": [
    "REQ-03",
    "REQ-07",
    "REQ-12"
  ],
  "decisiones": [
    "DEC-011",
    "DEC-012",
    "DEC-013"
  ],
  "codigo": {
    "commit": "8b8bdec34aadbb91d9b412da12cd8388193427fd",
    "notebook_sha256": "ce8824e860ae7e02684bf702fd6865b7d87cc9eb71917f56bfb0ee8d9ccb8e8a",
    "benchmark_sha256": "c40127fa1e03dfc156b792c997578014b589f6b0650b2a3fe0fb03462c918893",
    "notebook_entrega_plantilla": {
      "ruta": "resultados/preparacion/notebooks-entrega-v02/ASL_Secuencial.plantilla.ipynb",
      "sha256": "5c683a89f6b7a66699e33465307e5a27f16080aa0b0d8ea80118c1e7d92af61d"
    },
    "presentacion_por_fases": "AST del worker; threadpool_limits manteni

## 2. Datos y funciones originales utilizadas

Código original conservado: carga de imágenes y arquitectura. Las tablas muestran todos los conteos de clases y particiones.

In [4]:
def build_sampled_file_list(train_dir, classes, samples_per_class, seed=SEED):
    rng = random.Random(seed)
    records = []
    for c in classes:
        files = list((train_dir / c).glob('*.jpg'))
        chosen = rng.sample(files, min(samples_per_class, len(files)))
        for f in chosen:
            records.append({'filepath': str(f), 'label': c})
    return pd.DataFrame(records)

def load_images(df, img_size=IMG_SIZE):
    X = np.zeros((len(df), img_size, img_size, 3), dtype=np.float32)
    y = np.zeros(len(df), dtype=np.int64)
    for i, row in enumerate(df.itertuples()):
        img = Image.open(row.filepath).convert('RGB').resize((img_size, img_size))
        X[i] = np.asarray(img, dtype=np.float32) / 255.0
        y[i] = label_to_idx[row.label]
    return (X, y)

def build_cnn_a():
    model = keras.Sequential([layers.Input((IMG_SIZE, IMG_SIZE, 3)), layers.Conv2D(32, 3, activation='relu'), layers.MaxPooling2D(), layers.Conv2D(64, 3, activation='relu'), layers.MaxPooling2D(), layers.Flatten(), layers.Dense(128, activation='relu'), layers.Dropout(0.3), layers.Dense(NUM_CLASSES, activation='softmax')], name='CNN_A_baseline')
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

def build_cnn_b():

    def bn():
        return layers.BatchNormalization(momentum=0.9)
    model = keras.Sequential([layers.Input((IMG_SIZE, IMG_SIZE, 3)), layers.Conv2D(32, 3, padding='same', activation='relu'), bn(), layers.Conv2D(32, 3, padding='same', activation='relu'), layers.MaxPooling2D(), layers.Dropout(0.25), layers.Conv2D(64, 3, padding='same', activation='relu'), bn(), layers.Conv2D(64, 3, padding='same', activation='relu'), layers.MaxPooling2D(), layers.Dropout(0.25), layers.Conv2D(128, 3, padding='same', activation='relu'), bn(), layers.MaxPooling2D(), layers.Dropout(0.3), layers.Flatten(), layers.Dense(256, activation='relu'), bn(), layers.Dropout(0.4), layers.Dense(NUM_CLASSES, activation='softmax')], name='CNN_B_regularizada')
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

def build_fc(hidden_layers, dropout=0.3, name='FC'):
    model = keras.Sequential([layers.Input((IMG_SIZE, IMG_SIZE, 3)), layers.Flatten(), layers.BatchNormalization()], name=name)
    for units in hidden_layers:
        model.add(layers.Dense(units, activation='relu'))
        model.add(layers.Dropout(dropout))
    model.add(layers.Dense(NUM_CLASSES, activation='softmax'))
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

In [5]:
datos_manifest = pd.read_csv(ROOT / "resultados/secuencial/EXP-002/manifest_splits.csv", keep_default_na=False)
display(pd.crosstab(datos_manifest.clase, datos_manifest.split, margins=True))
display(datos_manifest.head(12))
print("17,400 rutas/identidades completas en el manifiesto congelado; no se vuelve a muestrear.")


split,test,train,val,All
clase,,,,
A,90,420,90,600
B,90,420,90,600
C,90,420,90,600
D,90,420,90,600
E,90,420,90,600
F,90,420,90,600
G,90,420,90,600
H,90,420,90,600
I,90,420,90,600


,dataset_id,ruta_relativa,sha256,clase,label_idx,split,orden
0,DAT-03-EXP-002,asl_alphabet_train/asl_alphabet_train/G/G655.jpg,c0bd22cd6c98e4c44f55768591883b3f09de1ea56a4d5086e1588f9c6bba1408,G,6,train,0
1,DAT-03-EXP-002,asl_alphabet_train/asl_alphabet_train/U/U872.jpg,99f8225febb986777b61f51b64a28dbe383ed4207961f574b12fc05ab16dcfd9,U,20,train,1
2,DAT-03-EXP-002,asl_alphabet_train/asl_alphabet_train/del/del1194.jpg,f1c0ce06721a2800426c606c91cf5ef7e1a8eda6b64e6a1d7fce5fd5241674fd,del,26,train,2
3,DAT-03-EXP-002,asl_alphabet_train/asl_alphabet_train/X/X1315.jpg,8aa9f39fb29eb8d6a3b3d08b8b2798ec2a87ae9d0acdefd00b7b4ab04c1b777f,X,23,train,3
4,DAT-03-EXP-002,asl_alphabet_train/asl_alphabet_train/L/L1488.jpg,de633263269d5853ae4e0fb29cde3d8b7117757b87c7c00dfc80e0b3ad211824,L,11,train,4
5,DAT-03-EXP-002,asl_alphabet_train/asl_alphabet_train/Q/Q2458.jpg,f91ac91df6363041db78bed1b5f6965394ccd957d6ee7a26889f4fe1551726c3,Q,16,train,5
6,DAT-03-EXP-002,asl_alphabet_train/asl_alphabet_train/R/R1802.jpg,a0ed4e004c4ecd3aae62a9bfca0f17322cf9a0fd1582dd5fc10d378a469df3a2,R,17,train,6
7,DAT-03-EXP-002,asl_alphabet_train/asl_alphabet_train/F/F639.jpg,db1e01d002a4453522ed1da46a5c7d52dd17ab290943f908bd6e520d4f94c7ce,F,5,train,7
8,DAT-03-EXP-002,asl_alphabet_train/asl_alphabet_train/V/V2125.jpg,51bb2043c46f2e3cab613a68a5b582ff7191610a6b405ba5dd8f16b0f889cb65,V,21,train,8
9,DAT-03-EXP-002,asl_alphabet_train/asl_alphabet_train/P/P1458.jpg,b16b54cab32dbec8c12826f245e32f66a02fd5d1bed59affb41ba895232642f7,P,15,train,9


17,400 rutas/identidades completas en el manifiesto congelado; no se vuelve a muestrear.


## Configuración efectiva del dispositivo y verificación de entradas

In [6]:
config = json.loads(CONFIG.read_text(encoding='utf-8'))

if sha(NOTEBOOK) != config['codigo']['notebook_sha256'] or sha(__file__) != config['codigo']['benchmark_sha256']:
    raise ValueError('Source identity differs from frozen experiment.')

threads = config['cpu']['hilos_matematicos']

for key, value in {'CUDA_VISIBLE_DEVICES': '-1', 'OMP_NUM_THREADS': str(threads), 'MKL_NUM_THREADS': str(threads), 'OPENBLAS_NUM_THREADS': str(threads), 'TF_NUM_INTRAOP_THREADS': str(threads), 'TF_NUM_INTEROP_THREADS': '1', 'TF_DETERMINISTIC_OPS': '1', 'TF_ENABLE_ONEDNN_OPTS': '1'}.items():
    os.environ[key] = value

import numpy as np

import pandas as pd

import tensorflow as tf

from sklearn.decomposition import PCA

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, confusion_matrix

from threadpoolctl import threadpool_limits, threadpool_info

tf.config.set_visible_devices([], 'GPU')

tf.config.threading.set_intra_op_parallelism_threads(threads)

tf.config.threading.set_inter_op_parallelism_threads(1)

tf.config.experimental.enable_op_determinism()

tf.keras.mixed_precision.set_global_policy('float32')

if tf.config.get_visible_devices('GPU'):
    raise ValueError('GPU visible in CPU benchmark.')

namespace = original_namespace()

namespace.update(keras=tf.keras, layers=tf.keras.layers, NUM_CLASSES=29)

classes = list(json.loads((EXPERIMENT / 'label_mapping.json').read_text(encoding='utf-8')).values())

namespace['label_to_idx'] = {name: i for i, name in enumerate(classes)}

data_root = Path(config['datos']['ruta'])

if sha(EXPERIMENT / 'manifest_splits.csv') != config['datos']['manifest_splits_sha256']:
    raise ValueError('Split manifest changed.')

manifest = pd.read_csv(EXPERIMENT / 'manifest_splits.csv', keep_default_na=False)

frames = {}

for split in ('train', 'val', 'test'):
    frame = manifest[manifest.split == split].sort_values('orden').copy()
    frame['filepath'] = frame.ruta_relativa.map(lambda s: str(data_root / s))
    frame['label'] = frame.clase
    frames[split] = frame

for item in manifest.itertuples():
    if sha(data_root / item.ruta_relativa) != item.sha256:
        raise ValueError(f'Input changed: {item.ruta_relativa}')

record = Recorder(directory, config, repetition)

(directory / 'configuracion.json').write_bytes(CONFIG.read_bytes())

write_json(directory / 'hardware.json', dict(cpu=config['cpu'], tensorflow=tf.__version__, visibles=[str(d) for d in tf.config.get_visible_devices()], logical=[str(d) for d in tf.config.list_logical_devices()], test_device=tf.constant([1.0]).device, build=tf.sysconfig.get_build_info(), threadpools=threadpool_info(), variables={k: os.environ.get(k) for k in ('CUDA_VISIBLE_DEVICES', 'OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'TF_NUM_INTRAOP_THREADS', 'TF_NUM_INTEROP_THREADS', 'TF_DETERMINISTIC_OPS', 'TF_ENABLE_ONEDNN_OPTS')}))

models, warmup, initial = ({}, {}, {})

warm_x, warm_y = namespace['load_images'](frames['train'].head(148))

warm_vx, warm_vy = namespace['load_images'](frames['val'].head(128))

data_directory = ROOT / 'data' / EXPERIMENT.name / directory.name

data_directory.mkdir(parents=True, exist_ok=False)

def augmented(name):
    namespace['best_cnn_name'] = name
    tree = ast.parse(''.join(namespace['_notebook']['cells'][52]['source']))
    nodes = [node for node in tree.body if isinstance(node, ast.Assign) and all((not isinstance(t, ast.Name) or t.id != 'history_aug' for t in node.targets)) or (isinstance(node, ast.Expr) and isinstance(node.value, ast.Call) and isinstance(node.value.func, ast.Attribute) and (node.value.func.attr == 'compile'))]
    exec(compile(ast.Module(body=nodes, type_ignores=[]), 'original/celda-52-configuracion', 'exec'), namespace)
    return namespace['cnn_aug']

In [7]:
print((directory / "hardware.json").read_text(encoding="utf8"))

{
  "cpu": {
    "modelo": "AMD Ryzen 5 4600H with Radeon Graphics",
    "nucleos_fisicos": 6,
    "nucleos_logicos": 12,
    "ram_bytes": 16558346240,
    "hilos_matematicos": 6,
    "intra_op": 6,
    "inter_op": 1,
    "workers_preparacion": 1,
    "afinidad": "sin restriccion adicional",
    "gpu_visible": false
  },
  "tensorflow": "2.22.0-rc0",
  "visibles": [
    "PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')"
  ],
  "logical": [
    "LogicalDevice(name='/device:CPU:0', device_type='CPU')"
  ],
  "test_device": "/job:localhost/replica:0/task:0/device:CPU:0",
  "build": {
    "is_cuda_build": false,
    "is_rocm_build": false,
    "is_tensorrt_build": false,
    "msvcp_dll_names": "msvcp140.dll,msvcp140_1.dll"
  },
  "threadpools": [
    {
      "user_api": "blas",
      "internal_api": "openblas",
      "num_threads": 12,
      "prefix": "libscipy_openblas",
      "filepath": "C:\\Users\\David\\Documents\\2026\\S2\\PARALELA\\CUDA\\PROYECTO-CUDA\\proyectoCUDA-p

In [8]:
threadpool_guard = threadpool_limits(limits=6)
threadpool_guard.__enter__()
print("Límite de bibliotecas matemáticas: 6 hilos")


Límite de bibliotecas matemáticas: 6 hilos


## Calentamiento y restauración de estados

In [9]:
for name, seed in config['aleatoriedad']['semillas_modelos'].items():
    record.state('inicializacion_y_warmup', modelo=name)
    tf.keras.utils.set_random_seed(seed)
    start = time.perf_counter()
    if name == 'CNN-A':
        model = namespace['build_cnn_a']()
    elif name == 'CNN-B':
        model = namespace['build_cnn_b']()
    elif name == 'FC-A':
        model = namespace['build_fc']([128, 64], dropout=0.3, name='FC_A_pequena')
    elif name == 'FC-B':
        model = namespace['build_fc']([512, 256, 128], dropout=0.4, name='FC_B_grande')
    else:
        model = augmented(name.split('+')[0])
    model.jit_compile = False
    model.optimizer.build(model.trainable_variables)
    variables = list(model.variables) + list(model.optimizer.variables)
    saved = [v.numpy().copy() for v in variables]
    h = hashlib.sha256()
    for array in saved:
        h.update(str((array.shape, str(array.dtype))).encode())
        h.update(array.tobytes())
    fingerprint = h.hexdigest()
    state_path = data_directory / f'{name}.estado-inicial.npz'
    np.savez(state_path, *saved)
    model.fit(warm_x, warm_y, validation_data=(warm_vx, warm_vy), epochs=1, batch_size=128, shuffle=False, verbose=0)
    if len(variables) != len(list(model.variables) + list(model.optimizer.variables)):
        raise ValueError('Warmup created unexpected state; cannot restore exactly.')
    for variable, value in zip(variables, saved):
        variable.assign(value)
    if any((not np.array_equal(v.numpy(), a) for v, a in zip(variables, saved))):
        raise ValueError('Warmup state restoration failed.')
    initial[name] = dict(state_sha256=fingerprint, archivo=state_path.relative_to(ROOT).as_posix(), archivo_sha256=sha(state_path), variables=len(variables), arquitectura=model.to_json(), compile=model.get_compile_config())
    warmup[name] = time.perf_counter() - start
    models[name] = model
    del saved

write_json(directory / 'estados-iniciales.json', initial)

write_json(directory / 'warmup.json', warmup)

NameError: name 'datetime' is not defined

## Preparación de datos en CPU

In [ ]:
metrics, training_history, epoch_rows = ({}, {}, [])

inputs = {}

wall_start_utc, wall_start = (utc(), time.perf_counter_ns())

def load_all():
    for split in ('train', 'val', 'test'):
        inputs[split] = namespace['load_images'](frames[split])

record.measure('preparacion_cpu', load_all, muestras=17400)

x, y = inputs['train']

vx, vy = inputs['val']

tx, ty = inputs['test']

In [ ]:
for etiqueta, (imagenes, clases_y) in inputs.items():
    print(etiqueta, "X", imagenes.shape, imagenes.dtype, "y", clases_y.shape, clases_y.dtype,
          "rango", float(imagenes.min()), float(imagenes.max()))
fig, axes = plt.subplots(2,6,figsize=(12,4))
for i,ax in enumerate(axes.flat):
    ax.imshow(x[i]); ax.set_title(str(y[i])); ax.axis("off")
fig.suptitle("Muestras reales tras preparación CPU")
from registro_notebook import mostrar_figura
mostrar_figura(fig)


## Callbacks y entrenamiento/evaluación originales

In [ ]:
class Epochs(tf.keras.callbacks.Callback):

    def __init__(self, name):
        super().__init__()
        self.name = name
        self.validation = 0.0
        self.epoch_validation = 0.0

    def on_epoch_begin(self, epoch, logs=None):
        self.start = time.perf_counter()
        self.epoch_validation = 0.0
        record.state('entrenamiento', modelo=self.name, epoca=epoch + 1)

    def on_test_begin(self, logs=None):
        self.test_start = time.perf_counter()

    def on_test_end(self, logs=None):
        seconds = time.perf_counter() - self.test_start
        self.validation += seconds
        self.epoch_validation += seconds

    def on_epoch_end(self, epoch, logs=None):
        epoch_rows.append(dict(modelo=self.name, epoca=epoch + 1, duracion_s=time.perf_counter() - self.start, validacion_s=self.epoch_validation, **logs))
        print(f'EPOCH {self.name} {epoch + 1}/15 {logs}', flush=True)

def fit_evaluate(name):
    tf.keras.utils.set_random_seed(config['aleatoriedad']['semillas_modelos'][name])
    callback = Epochs(name)
    history = record.measure(f'entrenamiento_{name}', lambda: models[name].fit(x, y, validation_data=(vx, vy), epochs=15, batch_size=128, shuffle=True, callbacks=[callback], verbose=0), epocas=15, pasos=1440, muestras=12180 * 15, batch_size=128, motivo='Incluye validacion; callbacks y duraciones anidadas conservados.')
    training_history[name] = history.history
    record.add(f'validacion_anidada_{name}', callback.validation, '', '', epocas=15, muestras=2610 * 15, motivo='Subintervalo de fit: no sumar al total ni tratar como fase adicional.')
    metrics[name] = record.measure(f'evaluacion_{name}', lambda: models[name].evaluate(tx, ty, verbose=0, return_dict=True), muestras=2610)
    write_csv(directory / 'epocas.csv', epoch_rows)
    write_json(directory / 'metricas-parciales.json', metrics)

## Entrenamiento y evaluación CNN-A

In [ ]:
fit_evaluate('CNN-A')

In [ ]:
modelo_tabla = 'CNN-A'
display(pd.DataFrame(training_history[modelo_tabla], index=range(1,16)).rename_axis("epoca"))
print("Prueba:", metrics[modelo_tabla])
print("Arquitectura real:")
models[modelo_tabla].summary()
hist = pd.DataFrame(training_history[modelo_tabla])
fig, ax = plt.subplots(1,2,figsize=(11,3.8))
for name in ("accuracy","val_accuracy"): ax[0].plot(range(1,len(hist)+1),hist[name],label=name)
for name in ("loss","val_loss"): ax[1].plot(range(1,len(hist)+1),hist[name],label=name)
for a in ax: a.set_xlabel("Época"); a.legend(); a.grid(alpha=.15)
ax[0].set_ylabel("Accuracy"); ax[1].set_ylabel("Pérdida")
fig.suptitle(modelo_tabla); fig.tight_layout()
from registro_notebook import mostrar_figura
mostrar_figura(fig)


## Entrenamiento y evaluación CNN-B

In [ ]:
fit_evaluate('CNN-B')

In [ ]:
modelo_tabla = 'CNN-B'
display(pd.DataFrame(training_history[modelo_tabla], index=range(1,16)).rename_axis("epoca"))
print("Prueba:", metrics[modelo_tabla])
print("Arquitectura real:")
models[modelo_tabla].summary()
hist = pd.DataFrame(training_history[modelo_tabla])
fig, ax = plt.subplots(1,2,figsize=(11,3.8))
for name in ("accuracy","val_accuracy"): ax[0].plot(range(1,len(hist)+1),hist[name],label=name)
for name in ("loss","val_loss"): ax[1].plot(range(1,len(hist)+1),hist[name],label=name)
for a in ax: a.set_xlabel("Época"); a.legend(); a.grid(alpha=.15)
ax[0].set_ylabel("Accuracy"); ax[1].set_ylabel("Pérdida")
fig.suptitle(modelo_tabla); fig.tight_layout()
from registro_notebook import mostrar_figura
mostrar_figura(fig)


## Diagnóstico de la CNN seleccionada

In [ ]:
best_cnn = max(('CNN-A', 'CNN-B'), key=lambda name: metrics[name]['accuracy'])

probs = record.measure('diagnostico_cnn', lambda: models[best_cnn].predict(tx, verbose=0), muestras=2610)

confusion = confusion_matrix(ty, np.argmax(probs, axis=1))

## Entrenamiento y evaluación FC-A

In [ ]:
fit_evaluate('FC-A')

In [ ]:
modelo_tabla = 'FC-A'
display(pd.DataFrame(training_history[modelo_tabla], index=range(1,16)).rename_axis("epoca"))
print("Prueba:", metrics[modelo_tabla])
print("Arquitectura real:")
models[modelo_tabla].summary()
hist = pd.DataFrame(training_history[modelo_tabla])
fig, ax = plt.subplots(1,2,figsize=(11,3.8))
for name in ("accuracy","val_accuracy"): ax[0].plot(range(1,len(hist)+1),hist[name],label=name)
for name in ("loss","val_loss"): ax[1].plot(range(1,len(hist)+1),hist[name],label=name)
for a in ax: a.set_xlabel("Época"); a.legend(); a.grid(alpha=.15)
ax[0].set_ylabel("Accuracy"); ax[1].set_ylabel("Pérdida")
fig.suptitle(modelo_tabla); fig.tight_layout()
from registro_notebook import mostrar_figura
mostrar_figura(fig)


## Entrenamiento y evaluación FC-B

In [ ]:
fit_evaluate('FC-B')

In [ ]:
modelo_tabla = 'FC-B'
display(pd.DataFrame(training_history[modelo_tabla], index=range(1,16)).rename_axis("epoca"))
print("Prueba:", metrics[modelo_tabla])
print("Arquitectura real:")
models[modelo_tabla].summary()
hist = pd.DataFrame(training_history[modelo_tabla])
fig, ax = plt.subplots(1,2,figsize=(11,3.8))
for name in ("accuracy","val_accuracy"): ax[0].plot(range(1,len(hist)+1),hist[name],label=name)
for name in ("loss","val_loss"): ax[1].plot(range(1,len(hist)+1),hist[name],label=name)
for a in ax: a.set_xlabel("Época"); a.legend(); a.grid(alpha=.15)
ax[0].set_ylabel("Accuracy"); ax[1].set_ylabel("Pérdida")
fig.suptitle(modelo_tabla); fig.tight_layout()
from registro_notebook import mostrar_figura
mostrar_figura(fig)


## PCA: ajuste únicamente con train

In [ ]:
pca = PCA(n_components=150, random_state=42)

projections = {}

def project():
    projections['train'] = pca.fit_transform(x.reshape(len(x), -1))
    projections['val'] = pca.transform(vx.reshape(len(vx), -1))
    projections['test'] = pca.transform(tx.reshape(len(tx), -1))

record.measure('pca', project, muestras=17400)

## Random Forest: ajuste y validación

In [ ]:
forests = {}

forest_val = {}

for name, trees, depth in (('RF-A', 100, None), ('RF-B', 300, 30)):
    forest = RandomForestClassifier(n_estimators=trees, max_depth=depth, random_state=42, n_jobs=-1)
    record.measure(f'entrenamiento_{name}', lambda: forest.fit(projections['train'], y), muestras=12180)
    pred = record.measure(f'validacion_{name}', lambda: forest.predict(projections['val']), muestras=2610)
    forest_val[name] = accuracy_score(vy, pred)
    forests[name] = forest

best_rf = max(forest_val, key=forest_val.get)

rf_pred = record.measure('evaluacion_RF', lambda: forests[best_rf].predict(projections['test']), muestras=2610)

metrics[best_rf] = dict(accuracy=accuracy_score(ty, rf_pred), val_accuracy=forest_val[best_rf])

## Entrenamiento y evaluación con aumento

In [ ]:
aug_name = best_cnn + '+aug'

fit_evaluate(aug_name)

In [ ]:
modelo_tabla = aug_name
display(pd.DataFrame(training_history[modelo_tabla], index=range(1,16)).rename_axis("epoca"))
print("Prueba:", metrics[modelo_tabla])
print("Arquitectura real:")
models[modelo_tabla].summary()
hist = pd.DataFrame(training_history[modelo_tabla])
fig, ax = plt.subplots(1,2,figsize=(11,3.8))
for name in ("accuracy","val_accuracy"): ax[0].plot(range(1,len(hist)+1),hist[name],label=name)
for name in ("loss","val_loss"): ax[1].plot(range(1,len(hist)+1),hist[name],label=name)
for a in ax: a.set_xlabel("Época"); a.legend(); a.grid(alpha=.15)
ax[0].set_ylabel("Accuracy"); ax[1].set_ylabel("Pérdida")
fig.suptitle(modelo_tabla); fig.tight_layout()
from registro_notebook import mostrar_figura
mostrar_figura(fig)


## Selección e inferencia sobre fotografías propias

In [ ]:
registry = {name: models[name] for name in ('CNN-A', 'CNN-B', 'FC-A', 'FC-B', aug_name)}

registry[best_rf] = forests[best_rf]

final = max(metrics, key=lambda name: metrics[name]['accuracy'])

own_files = sorted((ROOT / 'LAB3-DATASCIENCE/imagenes_grupo').glob('*.*'))

from PIL import Image

own_results = []

def prepare_one(path):
    with Image.open(path) as image:
        return np.asarray(image.convert('RGB').resize((64, 64)), dtype=np.float32) / 255.0

def predict_one(name, array):
    if name.startswith('RF'):
        return int(registry[name].predict(pca.transform(array.reshape(1, -1)))[0])
    return int(np.argmax(registry[name].predict(array[np.newaxis, ...], verbose=0)[0]))

def own_inference():
    for path in own_files:
        pred = predict_one(final, prepare_one(path))
        own_results.append(dict(modelo=final, etapa='final', archivo=path.name, real=path.stem.split('_')[0], prediccion=classes[pred]))
    arrays = [prepare_one(path) for path in own_files]
    for name in registry:
        for path, array in zip(own_files, arrays):
            pred = predict_one(name, array)
            own_results.append(dict(modelo=name, etapa='todos', archivo=path.name, real=path.stem.split('_')[0], prediccion=classes[pred]))

record.measure('inferencia_grupo', own_inference, muestras=len(own_files), motivo='15+90 predicciones individuales; 30 preprocesamientos; sin renderizar figuras.')

wall_end, wall_end_utc = (time.perf_counter_ns(), utc())

record.add('pipeline_total', (wall_end - wall_start) / 1000000000.0, wall_start_utc, wall_end_utc, epocas=75, pasos=7200, muestras=17400, batch_size=128)

## Identidad de tensores y persistencia de resultados

In [ ]:
fingerprints = {}

for split, (array, labels) in inputs.items():
    if array.shape != (len(frames[split]), 64, 64, 3) or not np.isfinite(array).all():
        raise ValueError('Invalid prepared tensors.')
    fingerprints[split] = dict(shape=list(array.shape), dtype=str(array.dtype), images_sha256=hashlib.sha256(array.tobytes()).hexdigest(), labels_sha256=hashlib.sha256(labels.tobytes()).hexdigest(), min=float(array.min()), max=float(array.max()))

write_json(directory / 'tensores.json', fingerprints)

write_json(directory / 'historiales.json', training_history)

write_csv(directory / 'predicciones_grupo.csv', own_results)

write_csv(directory / 'confusion.csv', [dict(clase=classes[i], **{c: int(v) for c, v in zip(classes, row)}) for i, row in enumerate(confusion)])

write_json(directory / 'metricas.json', dict(modelos=metrics, mejor_cnn=best_cnn, mejor_rf=best_rf, modelo_final=final, pca_varianza=float(pca.explained_variance_ratio_.sum()), grupo_accuracy={name: sum((row['real'].upper() == row['prediccion'].upper() for row in own_results if row['etapa'] == 'todos' and row['modelo'] == name)) / len(own_files) for name in registry}))

model_paths = {}

for name in registry:
    if name.startswith('RF'):
        continue
    path = data_directory / f'{name}.keras'
    registry[name].save(path)
    model_paths[name] = dict(ruta=path.relative_to(ROOT).as_posix(), sha256=sha(path))

import joblib

for name, obj in ((best_rf, forests[best_rf]), ('PCA', pca)):
    path = data_directory / f'{name}.joblib'
    joblib.dump(obj, path)
    model_paths[name] = dict(ruta=path.relative_to(ROOT).as_posix(), sha256=sha(path))

write_json(directory / 'modelos-guardados.json', model_paths)

record.state('completada', pipeline_s=(wall_end - wall_start) / 1000000000.0)

## Resultados completos de esta ejecución

In [ ]:
print((directory / "metricas.json").read_text(encoding="utf8"))
display(pd.read_csv(directory / "tiempos.csv"))
display(pd.read_csv(directory / "epocas.csv"))


In [ ]:
confusion_df = pd.read_csv(directory / "confusion.csv", index_col=0)
display(confusion_df)
fig, ax = plt.subplots(figsize=(10,8)); im=ax.imshow(confusion_df.values,cmap="Blues")
ax.set(xticks=range(29),yticks=range(29),xticklabels=confusion_df.columns,yticklabels=confusion_df.index,
       xlabel="Predicción",ylabel="Clase real",title="Matriz de confusión de "+best_cnn)
ax.tick_params(axis="x",labelrotation=90);fig.colorbar(im,ax=ax);fig.tight_layout();mostrar_figura(fig)
propias = pd.read_csv(directory / "predicciones_grupo.csv")
display(propias)
print("Modelo final:",final,"; PCA varianza:",pca.explained_variance_ratio_.sum())
print((directory / "modelos-guardados.json").read_text(encoding="utf8"))


## Recursos utilizados, logs y cierre de la ejecución nueva

In [ ]:
monitor_stop.set(); monitor_thread.join(timeout=15)
threadpool_guard.__exit__(None,None,None)
recursos_actuales = pd.read_csv(directory / "recursos.csv")
display(recursos_actuales)
columnas_cpu=[c for c in recursos_actuales if c.startswith("cpu_logico_")]
fig, axes = plt.subplots(2,1,figsize=(12,6))
for col in columnas_cpu: axes[0].plot(recursos_actuales.transcurrido_s,recursos_actuales[col],label=col)
axes[0].set(ylabel="CPU por lógico (%)",title="Uso real registrado durante esta ejecución")
axes[0].legend(ncol=4,fontsize=7)
axes[1].plot(recursos_actuales.transcurrido_s,recursos_actuales.gpu_util_pct,label="Actividad GPU (%)")
axes[1].plot(recursos_actuales.transcurrido_s,recursos_actuales.gpu_mem_used_mib/4096*100,label="VRAM ocupada (%)")
axes[1].set(xlabel="Tiempo de proceso (s)",ylabel="Porcentaje");axes[1].legend();fig.tight_layout();mostrar_figura(fig)
for captura in sorted(directory.glob("captura-*.png")):
    display(Image(filename=str(captura)))
    print(captura.name, (captura.with_suffix(".json")).read_text(encoding="utf8"))
write_json(directory / "salida.json",dict(codigo=0,fin_utc=utc(),tipo="ejecucion_directa_notebook"))
print("EJECUCIÓN NUEVA COMPLETA:", directory.name)


## Cinco mediciones históricas del informe — registros íntegros

Esta sección es histórica y está separada de la corrida nueva anterior. Se muestran stdout/stderr completos,épocas,métricas y tiempos de cada RUN. No se vuelven a entrenar ni se mezclan con el promedio original.

### Registro histórico RUN-20261117T171942Z-cpu-01

In [ ]:
historico = ROOT / "resultados/secuencial/EXP-002/RUN-20261117T171942Z-cpu-01"
print("HISTÓRICO:",historico.name)
print((historico/"stdout.log").read_text(encoding="utf8"))
print("STDERR ORIGINAL:")
print((historico/"stderr.log").read_text(encoding="utf8"))
display(pd.read_csv(historico/"epocas.csv"))
display(pd.read_csv(historico/"tiempos.csv"))
print((historico/"metricas.json").read_text(encoding="utf8"))


### Registro histórico RUN-20261117T175407Z-cpu-02

In [ ]:
historico = ROOT / "resultados/secuencial/EXP-002/RUN-20261117T175407Z-cpu-02"
print("HISTÓRICO:",historico.name)
print((historico/"stdout.log").read_text(encoding="utf8"))
print("STDERR ORIGINAL:")
print((historico/"stderr.log").read_text(encoding="utf8"))
display(pd.read_csv(historico/"epocas.csv"))
display(pd.read_csv(historico/"tiempos.csv"))
print((historico/"metricas.json").read_text(encoding="utf8"))


### Registro histórico RUN-20261117T182835Z-cpu-03

In [ ]:
historico = ROOT / "resultados/secuencial/EXP-002/RUN-20261117T182835Z-cpu-03"
print("HISTÓRICO:",historico.name)
print((historico/"stdout.log").read_text(encoding="utf8"))
print("STDERR ORIGINAL:")
print((historico/"stderr.log").read_text(encoding="utf8"))
display(pd.read_csv(historico/"epocas.csv"))
display(pd.read_csv(historico/"tiempos.csv"))
print((historico/"metricas.json").read_text(encoding="utf8"))


### Registro histórico RUN-20261117T190250Z-cpu-04

In [ ]:
historico = ROOT / "resultados/secuencial/EXP-002/RUN-20261117T190250Z-cpu-04"
print("HISTÓRICO:",historico.name)
print((historico/"stdout.log").read_text(encoding="utf8"))
print("STDERR ORIGINAL:")
print((historico/"stderr.log").read_text(encoding="utf8"))
display(pd.read_csv(historico/"epocas.csv"))
display(pd.read_csv(historico/"tiempos.csv"))
print((historico/"metricas.json").read_text(encoding="utf8"))


### Registro histórico RUN-20261117T192509Z-cpu-05

In [ ]:
historico = ROOT / "resultados/secuencial/EXP-002/RUN-20261117T192509Z-cpu-05"
print("HISTÓRICO:",historico.name)
print((historico/"stdout.log").read_text(encoding="utf8"))
print("STDERR ORIGINAL:")
print((historico/"stderr.log").read_text(encoding="utf8"))
display(pd.read_csv(historico/"epocas.csv"))
display(pd.read_csv(historico/"tiempos.csv"))
print((historico/"metricas.json").read_text(encoding="utf8"))


## Comparación histórica del informe (no incluye la corrida nueva)

In [ ]:
display(pd.read_csv(ROOT/"resultados/paralelo/EXP-003/comparacion_cnn_b.csv"))
display(pd.read_csv(ROOT/"resultados/paralelo/EXP-003/speedup.csv"))
print("Solo CNN-B base comparable. Preparación1→6workers es mejoraCPU;referencia histórica,RNG distinto y sin afirmación de saturaciónSM/VRAM.")
